# LFQ Video VAE — comma2k19 training

Trains a Lookup-Free Quantization video autoencoder on comma2k19 driving footage at near-native resolution (center-cropped to 1152×864), with **16× spatial compression** and **MSE + VGG perceptual** reconstruction loss.

**Before running:** set `Runtime → Change runtime type → GPU` (A100 is best; T4 works at batch 1).

Token grid per clip: `T × 54 × 72` = ~3.9k tokens/frame at the 1152×864 crop.

In [ ]:
# Allocator config must be set before torch is imported anywhere.
# expandable_segments=True lets the caching allocator stretch existing
# segments instead of failing fragmented requests - cuts OOMs caused by
# allocation patterns rather than peak memory.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

## Config

Point `DATA_ROOT` at wherever the comma2k19 chunks live in your Drive. The dataset class will recursively find all `.mkv` files.

In [ ]:
from pathlib import Path

# --- paths (EDIT THESE) ---
DATA_ROOT = Path('/content/drive/MyDrive/comma2k19')
CKPT_DIR  = Path('/content/drive/MyDrive/lfq_ckpts')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# --- input ---
CROP_W, CROP_H = 1152, 864    # center-crop (native is 1164x874)
CLIP_T  = 8                   # frames per clip
FPS     = 20                  # comma2k19 capture rate
SEG_FRAMES = 1200             # ~1 minute per .mkv

# --- model (sized for ~95 GB of GPU memory) ---
HIDDEN_DIM      = 256         # widened from 128 - more encoder capacity
EMBED_DIM       = 64
CODEBOOK_DIM    = 14          # 2^14 = 16384 codes
NUM_DOWNSAMPLES = 4           # 2^4 = 16x spatial compression

# --- loss weights ---
ENTROPY_W    = 0.1            # entropy regularizer in LFQ (paper default)
PERCEPTUAL_W = 0.1            # VGG perceptual term on top of MSE
VGG_CROP     = 384            # random crop fed to VGG; bigger = more detail signal

# --- optimization ---
# At batch=24, hidden=256, T=8 the first-conv activation is ~24 GB at fp16 and
# peak memory around the encoder/decoder boundary lands at ~70 GB. If you see OOM,
# step BATCH_SIZE down to 16; if there's lots of headroom, bump to 32.
BATCH_SIZE  = 24
GRAD_ACCUM  = 1               # no longer needed at this batch size
LR          = 5e-4            # mild bump for the larger effective batch
NUM_STEPS   = 20_000
LOG_EVERY   = 50
SAVE_EVERY  = 2_000
NUM_WORKERS = 16              # ffmpeg-decode parallelism; needs to keep up at batch 24

# --- adversarial (VQ-GAN-style PatchGAN) ---
USE_GAN          = True       # set False to skip the discriminator entirely
GAN_WARMUP_STEPS = 5000       # disc is dormant before this step; generator trains as before
GAN_WEIGHT       = 0.1        # weight on adversarial term in the generator loss
DISC_LR          = 3e-4       # disc has its own optimizer
DISC_HIDDEN      = 64         # 3D PatchGAN starting channel count
DISC_LAYERS      = 3          # number of strided conv layers


## Model code

Clones the `vq-vae` branch of the repo into `/content/repo` so we can import `LFQVAE` from the canonical `scripts/lfq.py`. **Push your local changes before running** — Colab pulls whatever's on the remote branch.

In [ ]:
import os, subprocess
REPO_DIR = '/content/repo'
REPO_URL = 'https://github.com/IanPTan/comma-vc-dev.git'
REPO_BRANCH = 'vq-vae'
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '-b', REPO_BRANCH, '--depth', '1',
                           REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
!ls {REPO_DIR}/scripts | head

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import sys
SCRIPTS_DIR = f'{REPO_DIR}/scripts'
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

from lfq import LFQVAE

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

model = LFQVAE(
    hidden_dim=HIDDEN_DIM,
    embed_dim=EMBED_DIM,
    codebook_dim=CODEBOOK_DIM,
    num_downsamples=NUM_DOWNSAMPLES,
    entropy_loss_weight=ENTROPY_W,
    use_checkpoint=True,    # drops encoder/decoder activations; recomputed on bwd
).to(device)
# Chunk size for the [N, 2^codebook_dim] entropy softmax. At batch=24, T=8,
# 1152x864 we get N=746,496 tokens; 65536-token chunks keep peak at ~6 GB.
model.quantizer.entropy_chunk_size = 65536

n_params = sum(p.numel() for p in model.parameters())
lat_h = CROP_H // (2 ** NUM_DOWNSAMPLES)
lat_w = CROP_W // (2 ** NUM_DOWNSAMPLES)
print(f'params: {n_params:,}')
print(f'codebook: 2^{CODEBOOK_DIM} = {2**CODEBOOK_DIM:,} codes')
print(f'tokens per clip: {CLIP_T} x {lat_h} x {lat_w} = {CLIP_T * lat_h * lat_w:,}')

## Dataset

Streams T-frame clips from random `.mkv` segments via `ffmpeg -ss`. Each DataLoader worker spawns its own ffmpeg subprocess, so I/O parallelism scales with `NUM_WORKERS`. Failed reads (corrupt segment, short file) silently skip to the next file.

In [ ]:
import subprocess
import random
import numpy as np
from torch.utils.data import IterableDataset, DataLoader


class CommaClipDataset(IterableDataset):
    """Yields (C, T, H, W) clips in [0, 1]."""

    def __init__(self, mkv_paths, clip_length=8, crop_h=864, crop_w=1152,
                 fps=20, seg_frames=1200):
        self.mkvs = [str(p) for p in mkv_paths]
        self.clip_length = clip_length
        self.crop_h = crop_h
        self.crop_w = crop_w
        self.fps = fps
        self.seg_frames = seg_frames

    def _read_clip(self, mkv_path, start_frame):
        start_time = start_frame / self.fps
        cmd = [
            'ffmpeg', '-hide_banner', '-loglevel', 'error',
            '-ss', f'{start_time:.3f}',
            '-i', mkv_path,
            '-frames:v', str(self.clip_length),
            '-vf', f'crop={self.crop_w}:{self.crop_h}',
            '-pix_fmt', 'rgb24',
            '-f', 'rawvideo', 'pipe:1',
        ]
        try:
            raw = subprocess.check_output(cmd, stderr=subprocess.DEVNULL, timeout=30)
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
            return None
        arr = np.frombuffer(raw, dtype=np.uint8).copy()
        try:
            arr = arr.reshape(-1, self.crop_h, self.crop_w, 3)
        except ValueError:
            return None
        if arr.shape[0] != self.clip_length:
            return None
        return arr

    def __iter__(self):
        info = torch.utils.data.get_worker_info()
        seed = info.seed if info is not None else 0
        rng = random.Random(seed)
        while True:
            mkv = rng.choice(self.mkvs)
            start = rng.randint(0, max(0, self.seg_frames - self.clip_length))
            arr = self._read_clip(mkv, start)
            if arr is None:
                continue
            x = torch.from_numpy(arr).float() / 255.0      # (T, H, W, C)
            yield x.permute(3, 0, 1, 2).contiguous()       # (C, T, H, W)


# discover all .mkv segments under DATA_ROOT
mkv_paths = sorted(DATA_ROOT.rglob('*.mkv'))
assert mkv_paths, f'no .mkv files under {DATA_ROOT} - check the path'
print(f'found {len(mkv_paths)} .mkv segments')

# small holdout for visualization
random.seed(0)
random.shuffle(mkv_paths)
val_paths   = mkv_paths[:max(1, len(mkv_paths) // 50)]
train_paths = mkv_paths[len(val_paths):]
print(f'train: {len(train_paths)} segments | val: {len(val_paths)}')

train_ds = CommaClipDataset(train_paths, CLIP_T, CROP_H, CROP_W, FPS, SEG_FRAMES)
val_ds   = CommaClipDataset(val_paths,   CLIP_T, CROP_H, CROP_W, FPS, SEG_FRAMES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=1, pin_memory=True)

## Perceptual loss

VGG16 features at relu1_2 / relu2_2 / relu3_3 — pulls detail back into the reconstructions that pure MSE would blur away.

VGG is applied per-frame on a random 256×256 crop (same crop for `x` and `recon`). The full-resolution VGG forward at 1152×864 would be ~2 GB of activations per layer; the random crop covers the whole image stochastically across training while keeping the memory footprint flat.

In [ ]:
from torchvision.models import vgg16, VGG16_Weights


class VGGPerceptualLoss(nn.Module):
    def __init__(self, layer_indices=(3, 8, 15), crop_size=VGG_CROP):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.DEFAULT).features.eval()
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg
        self.layer_indices = set(layer_indices)
        self.max_layer = max(layer_indices)
        self.crop_size = crop_size
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x, y):
        # x, y: (B, C, T, H, W) in [0, 1]
        B, C, T, H, W = x.shape
        x = x.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)
        y = y.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)

        cs = self.crop_size
        if H > cs and W > cs:
            top  = torch.randint(0, H - cs + 1, (1,)).item()
            left = torch.randint(0, W - cs + 1, (1,)).item()
            x = x[:, :, top:top + cs, left:left + cs]
            y = y[:, :, top:top + cs, left:left + cs]

        x = (x.clamp(0, 1) - self.mean) / self.std
        y = (y.clamp(0, 1) - self.mean) / self.std

        loss = 0.0
        for i, layer in enumerate(self.vgg):
            x = layer(x)
            y = layer(y)
            if i in self.layer_indices:
                loss = loss + F.mse_loss(x, y)
            if i >= self.max_layer:
                break
        return loss


perceptual = VGGPerceptualLoss().to(device)

## Discriminator (PatchGAN, 3D)

Small fully-convolutional 3D discriminator. Each output position classifies a ~70x70 spatial x 3-frame patch as real/fake. Hinge loss + spectral norm — the modern stable-GAN recipe.

The discriminator stays dormant for the first `GAN_WARMUP_STEPS` so the generator can finish its basic reconstruction learning before the GAN objective kicks in. Trying to train the GAN against a half-baked generator produces useless gradients.

In [ ]:
class VideoPatchDiscriminator(nn.Module):
    """3D PatchGAN with spectral normalization on every conv."""

    def __init__(self, in_channels=3, hidden=DISC_HIDDEN, n_layers=DISC_LAYERS):
        super().__init__()
        sn = nn.utils.spectral_norm
        layers = [
            sn(nn.Conv3d(in_channels, hidden, (3, 4, 4), (1, 2, 2), (1, 1, 1))),
            nn.LeakyReLU(0.2, inplace=True),
        ]
        ch = hidden
        for i in range(1, n_layers):
            nxt = min(ch * 2, 256)
            layers += [
                sn(nn.Conv3d(ch, nxt, (3, 4, 4), (1, 2, 2), (1, 1, 1))),
                nn.GroupNorm(8, nxt),
                nn.LeakyReLU(0.2, inplace=True),
            ]
            ch = nxt
        layers.append(sn(nn.Conv3d(ch, 1, kernel_size=1)))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  # (B, 1, T, H/2^n, W/2^n)


if USE_GAN:
    disc = VideoPatchDiscriminator().to(device)
    opt_d = torch.optim.Adam(disc.parameters(), lr=DISC_LR, betas=(0.5, 0.9))
    n_disc_params = sum(p.numel() for p in disc.parameters())
    print(f'discriminator params: {n_disc_params:,}')
    print(f'gan turns on at step {GAN_WARMUP_STEPS} with weight {GAN_WEIGHT}')
else:
    disc = None
    opt_d = None
    print('USE_GAN=False, training without discriminator')

## Train

Mixed-precision (fp16) + gradient clipping. Logs codebook utilization alongside losses — watch this to catch collapse early.

Two optimizers: `opt` updates the LFQVAE every step; `opt_d` updates the discriminator only after `GAN_WARMUP_STEPS`. Before warmup the loop is identical to the pre-GAN version (so resuming from a non-GAN checkpoint is seamless).

Auto-resumes from the latest checkpoint in `CKPT_DIR` if you re-run the cell. Discriminator state is loaded if present in the checkpoint; otherwise it starts fresh.

In [ ]:
import time
from tqdm.auto import tqdm

opt = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler()

# resume support — loads disc state if the checkpoint has it
start_step = 0
ckpts = sorted(CKPT_DIR.glob('ckpt_*.pt'))
if ckpts:
    state = torch.load(ckpts[-1], map_location=device)
    model.load_state_dict(state['model'])
    opt.load_state_dict(state['opt'])
    if USE_GAN and 'disc' in state:
        disc.load_state_dict(state['disc'])
        opt_d.load_state_dict(state['opt_d'])
        print(f"  also restored discriminator state")
    elif USE_GAN:
        print(f"  discriminator initialized fresh (checkpoint pre-dates GAN)")
    start_step = state['step']
    print(f'resumed from {ckpts[-1].name} at step {start_step}')
else:
    print('no checkpoint found, starting fresh')

loader_iter = iter(train_loader)
log = {'step': [], 'recon': [], 'perc': [], 'q': [], 'adv': [], 'd': [], 'used': []}
best_recon = float('inf')

model.train()
if USE_GAN: disc.train()
t0 = time.time()
step = start_step
pbar = tqdm(initial=start_step, total=NUM_STEPS, desc='training', dynamic_ncols=True)
while step < NUM_STEPS:
    gan_on = USE_GAN and step >= GAN_WARMUP_STEPS

    # --- Generator (LFQVAE) update ---
    opt.zero_grad(set_to_none=True)
    recon_acc = perc_acc = q_acc = adv_acc = 0.0
    for _ in range(GRAD_ACCUM):
        batch = next(loader_iter).to(device, non_blocking=True)
        with torch.cuda.amp.autocast(dtype=torch.float16):
            recon, q_loss, _ = model(batch)
            mse  = F.mse_loss(recon, batch)
            perc = perceptual(recon, batch)
            if gan_on:
                d_fake_g = disc(recon)
                adv = -d_fake_g.mean()
            else:
                adv = torch.zeros((), device=device)
            gen_total = (mse + PERCEPTUAL_W * perc + q_loss
                         + (GAN_WEIGHT * adv if gan_on else 0.0)) / GRAD_ACCUM
        scaler.scale(gen_total).backward()
        recon_acc += mse.item()
        perc_acc  += perc.item()
        q_acc     += q_loss.item()
        adv_acc   += adv.item() if gan_on else 0.0
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(opt)
    scaler.update()

    # --- Discriminator update (only after warmup) ---
    d_loss_val = 0.0
    if gan_on:
        opt_d.zero_grad(set_to_none=True)
        with torch.no_grad():
            with torch.cuda.amp.autocast(dtype=torch.float16):
                recon_for_d, _, _ = model(batch)
        with torch.cuda.amp.autocast(dtype=torch.float16):
            d_real = disc(batch)
            d_fake = disc(recon_for_d)
            d_loss = (F.relu(1.0 - d_real).mean() +
                      F.relu(1.0 + d_fake).mean())
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(disc.parameters(), max_norm=1.0)
        opt_d.step()
        d_loss_val = d_loss.item()

    if step % LOG_EVERY == 0:
        used, total_codes = model.quantizer.codebook_usage()
        postfix = {
            'recon': f'{recon_acc/GRAD_ACCUM:.4f}',
            'perc':  f'{perc_acc/GRAD_ACCUM:.4f}',
            'q':     f'{q_acc/GRAD_ACCUM:.4f}',
            'code%': f'{100*used/total_codes:.1f}',
        }
        if gan_on:
            postfix['adv'] = f'{adv_acc/GRAD_ACCUM:.4f}'
            postfix['d']   = f'{d_loss_val:.4f}'
        pbar.set_postfix(postfix)
        log['step'].append(step)
        log['recon'].append(recon_acc/GRAD_ACCUM)
        log['perc'].append(perc_acc/GRAD_ACCUM)
        log['q'].append(q_acc/GRAD_ACCUM)
        log['adv'].append(adv_acc/GRAD_ACCUM)
        log['d'].append(d_loss_val)
        log['used'].append(used)

    # best-recon snapshot (smoothed via single-step value; skip warmup noise)
    if step > 200 and (recon_acc / GRAD_ACCUM) < best_recon:
        best_recon = recon_acc / GRAD_ACCUM
        snap = {'step': step + 1, 'model': model.state_dict(),
                'opt': opt.state_dict(), 'recon': best_recon,
                'config': dict(hidden_dim=HIDDEN_DIM, embed_dim=EMBED_DIM,
                               codebook_dim=CODEBOOK_DIM, num_downsamples=NUM_DOWNSAMPLES)}
        if USE_GAN:
            snap['disc'] = disc.state_dict()
            snap['opt_d'] = opt_d.state_dict()
        torch.save(snap, CKPT_DIR / 'ckpt_best.pt')

    if (step + 1) % SAVE_EVERY == 0:
        ckpt_path = CKPT_DIR / f'ckpt_{step+1:06d}.pt'
        snap = {'step': step + 1, 'model': model.state_dict(),
                'opt': opt.state_dict(),
                'config': dict(hidden_dim=HIDDEN_DIM, embed_dim=EMBED_DIM,
                               codebook_dim=CODEBOOK_DIM, num_downsamples=NUM_DOWNSAMPLES)}
        if USE_GAN:
            snap['disc'] = disc.state_dict()
            snap['opt_d'] = opt_d.state_dict()
        torch.save(snap, ckpt_path)
        tqdm.write(f'  saved {ckpt_path.name}')

    pbar.update(1)
    step += 1
pbar.close()

print(f'\ndone in {(time.time() - t0) / 60:.1f} min')

## Visualize reconstructions

Pulls one val clip, runs forward, and shows original vs reconstruction across the T frames. Crops a 512×512 center patch for legibility — the full 1152×864 strip would be too wide for a notebook display.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model.eval()
batch = next(iter(val_loader)).to(device)
with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=torch.float16):
        recon, _, tokens = model(batch)

orig = batch[0].clamp(0, 1).float().cpu()
rec  = recon[0].clamp(0, 1).float().cpu()

# crop a center 512x512 patch for display
ch, cw = 512, 512
h0 = (orig.shape[2] - ch) // 2
w0 = (orig.shape[3] - cw) // 2
orig = orig[:, :, h0:h0+ch, w0:w0+cw]
rec  = rec[:,  :, h0:h0+ch, w0:w0+cw]

T = orig.shape[1]
fig, axes = plt.subplots(2, T, figsize=(T * 2.2, 4.4))
for t in range(T):
    axes[0, t].imshow(orig[:, t].permute(1, 2, 0).numpy())
    axes[0, t].axis('off')
    axes[1, t].imshow(rec[:, t].permute(1, 2, 0).numpy())
    axes[1, t].axis('off')
axes[0, 0].set_title('orig', loc='left')
axes[1, 0].set_title('recon', loc='left')
plt.tight_layout()
plt.show()

mse = F.mse_loss(recon.float(), batch.float()).item()
psnr = 10 * np.log10(1.0 / max(mse, 1e-10))
print(f'val mse: {mse:.5f}  psnr: {psnr:.2f} dB  tokens: {tuple(tokens.shape)}')